## Setup

Change that following variable settings match your deployed model's *Inference endpoint*. for example: 

```
vllm_endpoint = "https://model-vllm.apps.clusterx.sandboxx.opentlc.com"
```

In [ ]:
vllm_endpoint = "https://multinode-vllm-vllm-multinode.apps.cluster-rdl66.rdl66.sandbox743.opentlc.com"
api_key = None

## Chat completion with Requests library

Build and submit the REST request.

In [ ]:
import requests

def get_model(endpoint, token = None):
    models_endpoint = f"{endpoint}/v1/models"
    headers = {}
    if token:
        headers = {
            "Authorization": f"Bearer {token}"
        }
    response = requests.get(models_endpoint, headers=headers)
    model = response.json()["data"][0]["id"]
    return model

def completion_request(prompt, model, endpoint, token=None):
    completion_endpoint = f"{endpoint}/v1/chat/completions"

    headers = {
        "Content-Type": "application/json"
    }
    if token:
        headers["Authorization"] = f"Bearer {token}"

    json_data = {
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": 512,
        "temperature": 1,
        "top_p": 1,
        "n": 1,
        "stream": False,
        "presence_penalty": 0,
        "frequency_penalty": 0,
        "stop": ["string"],
        "user": "string"
    }

    response = requests.post(
        completion_endpoint,
        json=json_data,
        verify=True,
        headers=headers
    )

    return response.json()

In [ ]:
model = get_model(vllm_endpoint)
print(model)

In [ ]:
prediction = completion_request("What is AI?", model, vllm_endpoint)
print(prediction["choices"][0]["message"]["content"])

## Chat completion with OpenAI library

In [ ]:
!pip install openai

In [4]:
from openai import OpenAI

client = OpenAI(base_url=f"{vllm_endpoint}/v1", api_key="")

In [15]:
def get_model_openai(client):
    return client.models.list().data[0].id

def completion_openai(prompt, model, client: OpenAI):
    chat = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=100,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
    )
    return chat.choices[0].message.content

In [ ]:
model = get_model_openai(client)
model

In [ ]:
prompt = "What is AI?"
response = completion_openai(prompt, model, client)
print(response)